# Attention PPO Baseline - OPEN THIS FRESH COPY

This is a fresh notebook copy created to bypass the stale notebook tab/cache issue.

It includes `reward_config`, `speed_config`, and `traffic_config` with `vehicles_density`.

Open this file if the original notebook tab is still not showing the latest config block.

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from argparse import Namespace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import PPO

warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*")


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "ppo"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "ppo" / "attention_ppo"
SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "Elurant_PPO"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import attention_ppo

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook dir:", NOTEBOOK_DIR)
print("Script dir:", SCRIPT_DIR)
print("Results dir:", RESULTS_DIR)


## Config

This notebook now exposes the environment reward, speed, and traffic settings directly, similar to `PPO_trials.ipynb`.

Notes:
- `reward_config` is applied directly to the native `highway-v0` reward terms.
- `reward_speed_range`, `speed_limit`, and `vehicles_density` are applied directly.
- `ego_speed_range` and `other_speed_range` are kept here for experiment tracking parity with `PPO_trials.ipynb`; native `highway-v0` does not use them in the same custom way unless we later wrap or replace the env.

In [ ]:
# CODEX CONFIG MARKER - REWARD + SPEED + TRAFFIC CONFIG
args = Namespace(
    mode='train',
    timesteps=7680,
    n_envs=24,
    n_steps=32,
    batch_size=256,
    n_epochs=3,
    learning_rate=1e-3,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    seed=42,
    device='cpu',
    feature_size=32,
    attention_heads=1,
    embedding_width=32,
    torch_threads=1,
    model_path=str(RESULTS_DIR / 'ppo_highway_attention'),
    tensorboard_log=str(RESULTS_DIR / 'tensorboard'),
    episodes=8,
    render_mode='none',
)

reward_config = {
    'collision_reward': -1.0,
    'right_lane_reward': 0.1,
    'high_speed_reward': 0.4,
    'lane_change_reward': 0.0,
    'normalize_reward': True,
}

speed_config = {
    'reward_speed_range': [27.0, 30.0],
    'speed_limit': 30.0,
    'ego_speed_range': [27.0, 30.0],
    'other_speed_range': [15.0, 20.0],
}

traffic_config = {
    'vehicles_density': 2.0,
}

env_config = dict(attention_ppo.DEFAULT_ENV_CONFIG)
env_config.update(reward_config)
env_config.update({
    'reward_speed_range': speed_config['reward_speed_range'],
    'speed_limit': speed_config['speed_limit'],
    'ego_speed_range': speed_config['ego_speed_range'],
    'other_speed_range': speed_config['other_speed_range'],
    'vehicles_density': traffic_config['vehicles_density'],
})

print(json.dumps(vars(args), indent=2))
print('reward_config=', json.dumps(reward_config, indent=2))
print('speed_config=', json.dumps(speed_config, indent=2))
print('traffic_config=', json.dumps(traffic_config, indent=2))
print(f'rollout_size={args.n_envs * args.n_steps} steps/update')


## Train

In [ ]:
attention_ppo.DEFAULT_ENV_CONFIG = dict(env_config)
print('Using notebook env_config:')
print(json.dumps(env_config, indent=2))
attention_ppo.train(args)


## Load The Trained Model

In [ ]:
model_zip = Path(f'{args.model_path}.zip')
if not model_zip.exists():
    raise FileNotFoundError(f'Model not found yet: {model_zip}')

model = PPO.load(str(Path(args.model_path)), device=args.device)
print('Loaded model from', model_zip)


## Metrics Helpers

Notes:
- `collision_rate` is reported as the running percentage of evaluation episodes that ended in a crash.
- `overtakes` is a notebook-defined metric that counts vehicles that move from ahead of the ego vehicle to behind it during an episode.
- `TTC` is the same-lane time-to-collision against the closest slower vehicle ahead, clipped at `10` seconds.

In [ ]:
TTC_CAP = 10.0

def ego_lane_index(env) -> int:
    return int(env.unwrapped.vehicle.lane_index[2])

def ego_speed(env) -> float:
    ego = env.unwrapped.vehicle
    if hasattr(ego, 'speed'):
        return float(ego.speed)
    return float(np.linalg.norm(ego.velocity))

def same_lane_ttc(env, ttc_cap: float = TTC_CAP) -> float:
    ego = env.unwrapped.vehicle
    ego_x = float(ego.position[0])
    ego_v = ego_speed(env)
    lane_index = ego_lane_index(env)
    best_ttc = ttc_cap

    for vehicle in env.unwrapped.road.vehicles:
        if vehicle is ego:
            continue
        try:
            vehicle_lane = int(vehicle.lane_index[2])
        except Exception:
            continue
        if vehicle_lane != lane_index:
            continue

        dx = float(vehicle.position[0] - ego_x)
        if dx <= 1e-6:
            continue

        if hasattr(vehicle, 'speed'):
            other_v = float(vehicle.speed)
        else:
            other_v = float(np.linalg.norm(vehicle.velocity))
        rel_v = ego_v - other_v
        if rel_v <= 1e-6:
            continue

        best_ttc = min(best_ttc, dx / rel_v)

    return float(best_ttc)

def update_overtake_state(env, previous_rel_x: dict[int, float], overtaken_ids: set[int]) -> dict[int, float]:
    ego = env.unwrapped.vehicle
    ego_x = float(ego.position[0])
    current_rel_x: dict[int, float] = {}

    for vehicle in env.unwrapped.road.vehicles:
        if vehicle is ego:
            continue
        vehicle_id = id(vehicle)
        rel_x = float(vehicle.position[0] - ego_x)
        current_rel_x[vehicle_id] = rel_x
        prev_rel_x = previous_rel_x.get(vehicle_id)
        if prev_rel_x is not None and prev_rel_x > 0.0 and rel_x <= 0.0:
            overtaken_ids.add(vehicle_id)

    return current_rel_x

def evaluate_attention_model(model, episodes: int = 8, seed: int = 42, render_mode=None, config=None):
    resolved_config = dict(config or env_config)
    env = attention_ppo.make_highway_env(config=resolved_config, render_mode=render_mode)
    summaries = []
    traces = []

    for episode_idx in range(episodes):
        obs, info = env.reset(seed=seed + episode_idx)
        terminated = False
        truncated = False
        total_reward = 0.0
        previous_rel_x = {}
        overtaken_ids = set()

        speed_trace = []
        ttc_trace = []
        lane_trace = []
        reward_trace = []

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)

            previous_rel_x = update_overtake_state(env, previous_rel_x, overtaken_ids)
            speed_trace.append(float(info.get('speed', ego_speed(env))))
            ttc_trace.append(same_lane_ttc(env))
            lane_trace.append(ego_lane_index(env))
            reward_trace.append(float(reward))

        collision = bool(info.get('crashed', False))
        lane_changes = int(np.count_nonzero(np.diff(lane_trace))) if len(lane_trace) > 1 else 0

        summaries.append({
            'episode': int(episode_idx + 1),
            'reward': float(total_reward),
            'steps': int(len(speed_trace)),
            'collision': collision,
            'avg_speed': float(np.mean(speed_trace)) if speed_trace else 0.0,
            'max_speed': float(np.max(speed_trace)) if speed_trace else 0.0,
            'overtakes': int(len(overtaken_ids)),
            'avg_ttc': float(np.mean(ttc_trace)) if ttc_trace else TTC_CAP,
            'min_ttc': float(np.min(ttc_trace)) if ttc_trace else TTC_CAP,
            'lane_changes': lane_changes,
        })

        traces.append({
            'episode': int(episode_idx + 1),
            'speed': speed_trace,
            'ttc': ttc_trace,
            'lane': lane_trace,
            'reward': reward_trace,
        })

    env.close()
    return summaries, traces

def plot_attention_summary(summaries, save_path: Path | None = None):
    episodes = np.array([item['episode'] for item in summaries], dtype=int)
    rewards = np.array([item['reward'] for item in summaries], dtype=float)
    avg_speed = np.array([item['avg_speed'] for item in summaries], dtype=float)
    overtakes = np.array([item['overtakes'] for item in summaries], dtype=float)
    min_ttc = np.array([item['min_ttc'] for item in summaries], dtype=float)
    avg_ttc = np.array([item['avg_ttc'] for item in summaries], dtype=float)
    lane_changes = np.array([item['lane_changes'] for item in summaries], dtype=float)
    collisions = np.array([float(item['collision']) for item in summaries], dtype=float)
    running_collision_rate = 100.0 * np.cumsum(collisions) / np.arange(1, len(collisions) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    axes[0, 0].plot(episodes, rewards, marker='o')
    axes[0, 0].set_title('Episode Reward')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')

    axes[0, 1].plot(episodes, avg_speed, marker='o', color='tab:green')
    axes[0, 1].set_title('Average Speed')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('m/s')

    axes[0, 2].bar(episodes, overtakes, color='tab:orange')
    axes[0, 2].set_title('Overtakes Per Episode')
    axes[0, 2].set_xlabel('Episode')
    axes[0, 2].set_ylabel('Count')

    axes[1, 0].plot(episodes, running_collision_rate, marker='o', color='crimson')
    axes[1, 0].set_title('Running Collision Rate')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('%')
    axes[1, 0].set_ylim(0, 100)

    axes[1, 1].plot(episodes, min_ttc, marker='o', label='Min TTC')
    axes[1, 1].plot(episodes, avg_ttc, marker='o', label='Avg TTC')
    axes[1, 1].axhline(1.5, linestyle='--', color='red', alpha=0.6, label='1.5s warning')
    axes[1, 1].set_title('Time-To-Collision')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Seconds')
    axes[1, 1].legend()

    axes[1, 2].plot(episodes, lane_changes, marker='o', color='purple')
    axes[1, 2].set_title('Lane Changes')
    axes[1, 2].set_xlabel('Episode')
    axes[1, 2].set_ylabel('Count')

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print('Saved summary plot to', save_path)
    plt.show()

def plot_episode_trace(traces, episode_index: int = 0, save_path: Path | None = None):
    trace = traces[episode_index]
    steps = np.arange(1, len(trace['speed']) + 1)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

    axes[0].plot(steps, trace['speed'], color='tab:green')
    axes[0].set_ylabel('Speed (m/s)')
    axes[0].set_title(f"Episode {trace['episode']} Step Trace")

    axes[1].plot(steps, trace['ttc'], color='tab:blue')
    axes[1].axhline(1.5, linestyle='--', color='red', alpha=0.6)
    axes[1].set_ylabel('TTC (s)')

    axes[2].step(steps, trace['lane'], where='post', color='tab:purple')
    axes[2].set_ylabel('Lane')

    axes[3].plot(steps, trace['reward'], color='tab:orange')
    axes[3].set_ylabel('Reward')
    axes[3].set_xlabel('Step')

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print('Saved episode trace to', save_path)
    plt.show()


## Evaluate And Save Metrics

In [ ]:
summaries, traces = evaluate_attention_model(
    model,
    episodes=args.episodes,
    seed=args.seed,
    render_mode=None,
    config=env_config,
)

summary_path = RESULTS_DIR / 'attention_eval_summary.json'
summary_path.write_text(json.dumps(summaries, indent=2), encoding='utf-8')
print('Saved evaluation summary to', summary_path)

overall_metrics = {
    'episodes': len(summaries),
    'mean_reward': float(np.mean([item['reward'] for item in summaries])),
    'collision_rate_percent': float(100.0 * np.mean([float(item['collision']) for item in summaries])),
    'mean_avg_speed': float(np.mean([item['avg_speed'] for item in summaries])),
    'mean_overtakes': float(np.mean([item['overtakes'] for item in summaries])),
    'mean_min_ttc': float(np.mean([item['min_ttc'] for item in summaries])),
}
print(json.dumps(overall_metrics, indent=2))
summaries


## Plot Metrics

In [ ]:
plot_attention_summary(summaries, save_path=RESULTS_DIR / 'attention_eval_summary.png')
plot_episode_trace(traces, episode_index=0, save_path=RESULTS_DIR / 'attention_eval_episode_1.png')


attention_ppo.DEFAULT_ENV_CONFIG = dict(env_config)
eval_args = Namespace(**vars(args))
eval_args.mode = 'eval'
eval_args.render_mode = 'human'
attention_ppo.evaluate(eval_args)


In [ ]:
eval_args = Namespace(**vars(args))
eval_args.mode = 'eval'
eval_args.render_mode = 'human'
attention_ppo.evaluate(eval_args)
